<a href="https://colab.research.google.com/github/Jason980102/cloud-mediapipe-pose-analysis/blob/main/pose_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
VIDEO_ROOT = "/content/drive/MyDrive/taekwondo_videos"
OUTPUT_DIR = "/content/drive/MyDrive/taekwondo_keypoints_csv"

In [17]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
import os

ROOT = "/content/drive/MyDrive/taekwondo_videos"

def print_tree(start_path, max_depth=3):
    for root, dirs, files in os.walk(start_path):
        level = root.replace(start_path, "").count(os.sep)
        if level > max_depth:
            continue

        indent = " " * 4 * level
        print(f"{indent}📁 {os.path.basename(root)}")

        sub_indent = " " * 4 * (level + 1)
        for f in files:
            print(f"{sub_indent}📄 {f}")

print_tree(ROOT, max_depth=3)

📁 taekwondo_videos
    📁 20240902
        📁 U11313037(2號機)(5順位)
            📄 V_20240902_110058_ES0.mp4
            📄 V_20240902_110137_ES0.mp4
            📄 V_20240902_110029_ES0.mp4
            📄 V_20240902_110207_ES0.mp4
            📄 V_20240902_105959_ES0.mp4
            📄 V_20240902_105924_ES0.mp4
        📁 U11313034(2號機)(4順位)
            📄 V_20240902_105616_ES0.mp4
            📄 V_20240902_105504_ES0.mp4
            📄 V_20240902_105334_ES0.mp4
            📄 V_20240902_105645_ES0.mp4
            📄 V_20240902_105543_ES0.mp4
            📄 V_20240902_105429_ES0.mp4
        📁 U11213030(2號機)(2順位)
            📄 V_20240902_103955_ES0.mp4
            📄 V_20240902_103901_ES0.mp4
            📄 V_20240902_104123_ES0.mp4
            📄 V_20240902_104040_ES0.mp4
            📄 V_20240902_104232_ES0.mp4
            📄 V_20240902_104157_ES0.mp4
        📁 U11313035(2號機)(6順位)
            📄 V_20240902_110613_ES0.mp4
            📄 V_20240902_110343_ES0.mp4
            📄 V_20240902_110537_ES0.mp4
      

In [19]:
import os
import re
import pandas as pd

VIDEO_ROOT = "/content/drive/MyDrive/taekwondo_videos"
TARGET_TOP_FOLDERS = {"freshmen", "sophomore", "junior"}
VIDEO_EXTS = (".mp4", ".MP4", ".mov", ".MOV", ".avi", ".AVI", ".mkv", ".MKV")

def guess_action_label(file_name, path_text):
    text = f"{file_name} {path_text}"

    if "旋踢" in text:
        return "roundhouse_kick"
    elif "側踩" in text:
        return "side_kick"
    elif "下壓" in text:
        return "axe_kick"
    elif "後踢" in text:
        return "back_kick"
    elif "後旋" in text:
        return "spinning_hook_kick"
    elif "正拳" in text:
        return "straight_punch"
    else:
        return "unknown"

def is_two_minute_video(file_name, path_text):
    text = f"{file_name} {path_text}"
    keywords = ["兩分鐘", "2分鐘", "2 minute", "2-minute"]
    return any(k in text for k in keywords)

def extract_student_id(parts):
    for p in parts:
        m = re.search(r"(U\d{8})", p)
        if m:
            return m.group(1)
    return None

rows = []

for top_folder in TARGET_TOP_FOLDERS:
    start_path = os.path.join(VIDEO_ROOT, top_folder)
    if not os.path.exists(start_path):
        print(f"Folder not found: {start_path}")
        continue

    for root, dirs, files in os.walk(start_path):
        for f in files:
            if not f.endswith(VIDEO_EXTS):
                continue

            full_path = os.path.join(root, f)
            rel_path = os.path.relpath(full_path, VIDEO_ROOT)
            parts = rel_path.split(os.sep)

            rows.append({
                "grade_group": parts[0] if len(parts) > 0 else None,      # 大一 / 大二 / 大三
                "student_folder": parts[1] if len(parts) > 1 else None,   # U11313036
                "session_folder": parts[2] if len(parts) > 2 else None,   # 第四次訓練紀錄...
                "file_name": f,
                "file_ext": os.path.splitext(f)[1],
                "student_id": extract_student_id(parts),
                "full_path": full_path,
                "relative_path": rel_path,
                "is_two_minute": is_two_minute_video(f, rel_path),
                "action_label": guess_action_label(f, rel_path),
            })

df_manifest = pd.DataFrame(rows)

print("Total videos in 大一/大二/大三:", len(df_manifest))
df_manifest.head(20)

Folder not found: /content/drive/MyDrive/taekwondo_videos/大一
Folder not found: /content/drive/MyDrive/taekwondo_videos/大二
Folder not found: /content/drive/MyDrive/taekwondo_videos/大三
Total videos in 大一/大二/大三: 0


""
